# Fashion MNIST Dataset

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import torch

# Load CSV files
df_train = pd.read_csv("/Users/bekzodabdul/Desktop/DOCUMENTS/Machine Learning/Machine Learning in the Cloud and at the Edge/Project/fl_DiverseVul/flower/dataset/fashion-mnist_train.csv")
df_test = pd.read_csv("/Users/bekzodabdul/Desktop/DOCUMENTS/Machine Learning/Machine Learning in the Cloud and at the Edge/Project/fl_DiverseVul/flower/dataset/fashion-mnist_test.csv")

# Split labels and pixels
X_train = df_train.drop("label", axis=1).values
y_train = df_train["label"].values

X_test = df_test.drop("label", axis=1).values
y_test = df_test["label"].values

# Normalize pixel values (0-255 → 0.0-1.0)
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape to [N, 1, 28, 28]
X_train = X_train.reshape(-1, 1, 28, 28).astype("float32")
X_test = X_test.reshape(-1, 1, 28, 28).astype("float32")

In [2]:
class FashionCSVDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [3]:
def partition_data(X, y, num_clients=5):
    size = len(X) // num_clients
    shards = []
    for i in range(num_clients):
        x_shard = X[i * size : (i + 1) * size]
        y_shard = y[i * size : (i + 1) * size]
        shards.append(FashionCSVDataset(x_shard, y_shard))
    return shards

client_datasets = partition_data(X_train, y_train, num_clients=5)
test_dataset = FashionCSVDataset(X_test, y_test)

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))     # (batch, 32, 26, 26)
        x = F.relu(self.conv2(x))     # (batch, 64, 24, 24)
        x = F.max_pool2d(x, 2)        # (batch, 64, 12, 12)
        x = torch.flatten(x, 1)       # (batch, 9216)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [5]:
import flwr as fl
import torch
from torch.utils.data import DataLoader

class FashionClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, test_data):
        self.model = model
        self.train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        self.test_loader = DataLoader(test_data, batch_size=32)

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        for param, val in zip(self.model.state_dict().keys(), parameters):
            self.model.state_dict()[param].copy_(torch.tensor(val))

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)
        for _ in range(1):  # 1 local epoch
            for x, y in self.train_loader:
                optimizer.zero_grad()
                loss = F.cross_entropy(self.model(x), y)
                loss.backward()
                optimizer.step()
        return self.get_parameters(config), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        loss, correct = 0.0, 0
        total = 0
        with torch.no_grad():
            for x, y in self.test_loader:
                pred = self.model(x)
                loss += F.cross_entropy(pred, y, reduction='sum').item()
                correct += (pred.argmax(dim=1) == y).sum().item()
                total += y.size(0)
        return loss / total, total, {"accuracy": correct / total}

2025-07-25 11:55:02,862	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [6]:
# %pip install -U "flwr[simulation]"

def client_fn(cid: str):
    model = Net()
    cid_int = int(cid)
    return FashionClient(model, client_datasets[cid_int], test_dataset)

fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=5,
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=fl.server.strategy.FedAvg(),
)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2025-07-25 11:55:07,875	INFO worker.py:1771 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'CPU': 8.0, 'memory': 3419661927.0, 'node:127.0.0.1': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 1709830963.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      No `client_resources` specified. Using minimal resources for clients.
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpu

History (loss, distributed):
	round 1: 0.49162329745292666
	round 2: 0.31187353870868684
	round 3: 0.2730935081958771
	round 4: 0.25596926501989364
	round 5: 0.23722806186676026